In [11]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

In [12]:
DATASET_PATH = r"isolated_alphabets_per_user"
IMG_SIZE = 32

images = []
labels = []

for user_folder in os.listdir(DATASET_PATH):
    user_path = os.path.join(DATASET_PATH, user_folder)

    if os.path.isdir(user_path):

        for file in os.listdir(user_path):
            path = os.path.join(user_path, file)

            img = cv2.imread(path)

            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            _, thresh = cv2.threshold(
                gray, 0, 255,
                cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
            )

            img = cv2.resize(thresh, (IMG_SIZE, IMG_SIZE))

            images.append(img)

            # 🔥 label extraction from filename
            parts = file.split('_')
            label = parts[1] + "_" + parts[2]
            labels.append(label)

images = np.array(images)
labels = np.array(labels)

print("Loaded:", images.shape)

Loaded: (53199, 32, 32)


In [13]:
print(labels[:20])

['ain_begin' 'ain_begin' 'ain_begin' 'ain_begin' 'ain_begin' 'ain_begin'
 'ain_begin' 'ain_begin' 'ain_begin' 'ain_begin' 'ain_end' 'ain_end'
 'ain_end' 'ain_end' 'ain_end' 'ain_end' 'ain_end' 'ain_end' 'ain_end'
 'ain_end']


In [14]:
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

# save classes for later
np.save("label_classes.npy", le.classes_)

# normalize + reshape
images = images / 255.0
images = images.reshape(-1, 32, 32, 1)

# split
X_train, X_test, y_train, y_test = train_test_split(
    images, labels_encoded, test_size=0.2, random_state=42
)

In [15]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(32,32,1)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(le.classes_), activation='softmax')
])

In [16]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.1901 - loss: 3.1081 - val_accuracy: 0.4943 - val_loss: 1.8077
Epoch 2/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.4293 - loss: 1.9383 - val_accuracy: 0.5836 - val_loss: 1.4312
Epoch 3/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.5160 - loss: 1.5965 - val_accuracy: 0.6396 - val_loss: 1.2240
Epoch 4/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - accuracy: 0.5653 - loss: 1.4207 - val_accuracy: 0.6556 - val_loss: 1.1144
Epoch 5/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.6026 - loss: 1.2866 - val_accuracy: 0.6760 - val_loss: 1.0477
Epoch 6/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.6243 - loss: 1.1987 - val_accuracy: 0.6884 - val_loss: 0.9987
Epoch 7/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - accuracy: 0.6505 - loss: 1.1185 - val_accuracy: 0.6943 - val_loss: 0.9699
Epoch 8/10
1330/1330 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.6693 -

In [18]:
model.save("cnn_per_user.keras")
print("Model saved")

Model saved


In [27]:
import cv2
import numpy as np
import tensorflow as tf

# ===== LOAD MODEL =====
model = tf.keras.models.load_model("cnn_per_user.keras")
CLASSES = np.load("label_classes.npy")

IMAGE_PATH = "word_test.png"

# ===== READ IMAGE =====
img = cv2.imread(IMAGE_PATH)

if img is None:
    print("❌ Image not found")
    exit()

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# ===== THRESHOLD =====
_, thresh = cv2.threshold(
    gray, 0, 255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# ===== SIMPLE STABLE SEGMENTATION =====
h, w = thresh.shape

# automatic estimation of number of characters
num_chars = min(6, max(2, w // 120))

step = w // num_chars

segments = []
for i in range(num_chars):
    x1 = i * step
    x2 = (i+1) * step if i < num_chars-1 else w
    segments.append((x1, x2))

# Arabic direction (right → left)
segments = sorted(segments, key=lambda x: x[0], reverse=True)

predicted = []

# ===== PREDICTION =====
for i, (x1, x2) in enumerate(segments):

    char = thresh[:, x1:x2]

    if char.size == 0:
        continue

    char = cv2.resize(char, (32,32))
    char = char / 255.0
    char = char.reshape(1,32,32,1)

    pred = model.predict(char, verbose=0)

    idx = np.argmax(pred)
    confidence = np.max(pred)

    label = CLASSES[idx]
    base = label.split('_')[0]

    print(f"Segment {i+1}: {label} → {base} ({confidence:.2%})")

    if confidence > 0.4:
        predicted.append((base, confidence))

# ===== CLEAN OUTPUT =====
# ===== SORT BY CONFIDENCE =====
predicted = sorted(predicted, key=lambda x: x[1], reverse=True)

# keep only top 3 predictions
top_preds = predicted[:3]

# restore order (important for word)
top_preds = sorted(top_preds, key=lambda x: predicted.index(x))

# extract letters
cleaned = [p[0] for p in top_preds]

print("\nDetected Characters (approximate):")
for i, (char, conf) in enumerate(predicted):
    print(f"{i+1}. {char} ({conf:.2%})")

# ===== DISPLAY IMAGE =====
cv2.imshow("Input Image", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Segment 1: yaa_middle → yaa (41.63%)
Segment 2: beh_end → beh (52.13%)
Segment 3: beh_regular → beh (27.39%)
Segment 4: meem_middle → meem (22.22%)
Segment 5: jeem_begin → jeem (65.77%)
Segment 6: heh_end → heh (95.78%)

Detected Characters (approximate):
1. heh (95.78%)
2. jeem (65.77%)
3. beh (52.13%)
4. yaa (41.63%)
